# **Demo 01: Building an Agent With the OpenAI Agents SDK and Dynamic Web Search**


**Objective:** To demonstrate how to build a tool-enabled AI agent using the OpenAI Agents SDK. The agent autonomously decides when to call a real-time web search tool (Tavily) based on the user query, and produces a grounded, up-to-date response.

**Prerequisites:** OpenAI API key, Tavily API key

**Tools required:** Python

**Scenario:** A product manager at a digital knowledge platform needs to handle a high volume of user queries, many of which require real-time information. The goal is to automate query handling using the OpenAI Agents SDK — a lightweight, production-ready framework where an agent is given a set of tools and decides autonomously when and how to use them to answer each query accurately.

In [1]:
# Step 1: Install required libraries
#
# openai-agents  — the OpenAI Agents SDK; provides Agent, Runner, and @function_tool
# tavily-python  — Tavily's official client (installed for completeness; we call the REST API directly below)
# requests       — standard HTTP library used to call the Tavily search REST endpoint
# python-dotenv  — loads environment variables from a .env file into os.environ

!pip install openai-agents tavily-python requests python-dotenv


  Using cached openai_agents-0.17.7-py3-none-any.whl.metadata (9.1 kB)
  Using cached tavily_python-0.7.26-py3-none-any.whl.metadata (12 kB)
  Using cached requests-2.34.2-py3-none-any.whl.metadata (4.8 kB)
  Using cached python_dotenv-1.2.2-py3-none-any.whl.metadata (27 kB)
  Using cached griffelib-2.1.0-py3-none-any.whl.metadata (1.3 kB)
  Using cached mcp-1.28.0-py3-none-any.whl.metadata (9.4 kB)
  Using cached openai-2.44.0-py3-none-any.whl.metadata (34 kB)
  Using cached pydantic-2.13.4-py3-none-any.whl.metadata (109 kB)
  Using cached websockets-16.0-cp314-cp314-macosx_11_0_arm64.whl.metadata (6.8 kB)
  Using cached charset_normalizer-3.4.7-cp314-cp314-macosx_10_15_universal2.whl.metadata (40 kB)
  Using cached idna-3.18-py3-none-any.whl.metadata (6.1 kB)
  Using cached urllib3-2.7.0-py3-none-any.whl.metadata (6.9 kB)
  Using cached certifi-2026.6.17-py3-none-any.whl.metadata (2.5 kB)
  Using cached anyio-4.14.1-py3-none-any.whl.metadata (4.6 kB)
  Using cached httpx_sse-0.4.3-py

In [2]:
# Step 2: Configure the OpenAI client and import Agents SDK primitives
#
# The OpenAI Agents SDK is model-agnostic — it can use any OpenAI-compatible
# endpoint. Here we point it at the standard OpenAI API.

import os
import requests                        # used later to call the Tavily search API
from dotenv import load_dotenv         # reads key=value pairs from a .env file into os.environ
from openai import AsyncOpenAI         # async client: non-blocking, required by the Agents SDK
from agents import (
    Agent,                             # the core Agent class
    Runner,                            # orchestrates the agent loop (LLM → tools → LLM → ...)
    function_tool,                     # decorator that turns a Python function into an agent tool
    set_default_openai_client,         # tells the SDK which OpenAI client to use
    set_tracing_disabled,              # disables automatic LLM call tracing (keeps output clean)
)

# ── Load credentials from .env ────────────────────────────────────────────────
# load_dotenv() looks for a .env file in the current directory (or any parent)
# and loads its key=value pairs into the process environment.
# The actual secrets never appear in the notebook source.
load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

# Fail fast if either key is missing so the error is obvious.
assert OPENAI_API_KEY, "OPENAI_API_KEY not found — add it to your .env file"
assert TAVILY_API_KEY, "TAVILY_API_KEY not found — add it to your .env file"

# ── Wire up the SDK ───────────────────────────────────────────────────────────
# AsyncOpenAI wraps the OpenAI REST API in an async-friendly client.
client = AsyncOpenAI(api_key=OPENAI_API_KEY)

# set_default_openai_client registers this client as the one the SDK will use
# for every Agent and Runner call in this notebook.
set_default_openai_client(client)

# Tracing logs every LLM call to OpenAI's servers. We disable it here so the
# output stays focused on the agent's responses.
set_tracing_disabled(True)

print("OpenAI client configured.")


OpenAI client configured.


In [3]:
# Step 3: Define a web search tool using the @function_tool decorator
#
# KEY CONCEPT — Tools give agents the ability to act:
#   The @function_tool decorator wraps a regular Python function so the agent
#   can call it. The SDK automatically converts the function's name, parameters,
#   and docstring into a JSON schema that the LLM uses to decide WHEN and HOW
#   to invoke the tool.
#
#   When the LLM decides it needs current information, it emits a tool-call
#   request. The Runner intercepts it, calls search_web(), and feeds the result
#   back to the LLM so it can form a final answer.

@function_tool
def search_web(query: str) -> str:
    # This docstring is what the LLM reads to understand what this tool does.
    # Write it clearly — it directly influences the agent's decision to use the tool.
    """Search the web for recent or real-time information on a topic."""

    # Call the Tavily search REST API (https://tavily.com)
    url = "https://api.tavily.com/search"
    payload = {
        "api_key": TAVILY_API_KEY,
        "query": query,      # the search query the agent decided to run
        "max_results": 5,    # limit to 5 results to keep the context concise
    }
    response = requests.post(url, json=payload, timeout=30)
    response.raise_for_status()  # raise an exception if the API returned an error status

    data = response.json()

    # Format each result as "Title - URL" on its own line.
    # Returning plain text keeps the context simple; the LLM will summarise it.
    results = [
        f"{r.get('title', 'No title')} - {r.get('url', 'No URL')}"
        for r in data.get("results", [])
    ]
    return "\n".join(results) if results else "No results found."

print("Tool defined:", search_web.name)


Tool defined: search_web


In [4]:
# Step 4: Create the agent
#
# An Agent is the combination of:
#   • A system prompt (instructions) — sets the agent's persona and behaviour
#   • A model             — the LLM that reasons and decides what to do
#   • A tool list         — capabilities the agent can invoke autonomously
#
# The agent does NOT call tools blindly on every request. The LLM reads the
# query, considers its instructions, and decides whether a tool call is needed.
# This is the core of autonomous agent behaviour.

research_agent = Agent(
    name="FMCG Research Agent",

    # instructions = the system prompt. This shapes how the LLM behaves:
    # when to search, what tone to use, and what audience to target.
    instructions=(
        "You are a research assistant specialising in FMCG (Fast-Moving Consumer Goods). "
        "Use the search_web tool to retrieve current information whenever the query "
        "requires recent data, news, or statistics. "
        "Summarise findings clearly and concisely for a business audience."
    ),

    model="gpt-4o-mini",   # the LLM used for reasoning; swap for gpt-4o for higher quality

    tools=[search_web],    # tools the agent is allowed to call; add more tools to extend capability
)

print(f"Agent '{research_agent.name}' ready with {len(research_agent.tools)} tool(s).")


Agent 'FMCG Research Agent' ready with 1 tool(s).


In [5]:
# Step 5a: Run the agent on a query that does NOT require web search
#
# This query asks for general knowledge the LLM already has.
# Expected behaviour: the agent answers directly without calling search_web.
#
# Runner.run() manages the full agent loop:
#   1. Sends the query + system prompt to the LLM
#   2. If the LLM requests a tool call, Runner executes it and loops back
#   3. When the LLM returns a plain-text response (no tool call), the loop ends
#
# result.final_output contains the agent's last text message to the user.

simple_query = "What does FMCG stand for and what are typical product categories?"

result = await Runner.run(research_agent, simple_query)

print("[Query]:", simple_query)
print("\n[Agent Response]:")
print(result.final_output)


[Query]: What does FMCG stand for and what are typical product categories?

[Agent Response]:
FMCG stands for Fast-Moving Consumer Goods. These are products that sell quickly at relatively low cost. Typical product categories include:

1. **Food and Beverages**
   - Snacks, packaged foods, dairy products, soft drinks, and alcoholic beverages.

2. **Personal Care**
   - Toiletries (shampoo, soap, toothpaste), cosmetics, and grooming products.

3. **Household Care**
   - Cleaning products, detergents, and paper products (toilet paper, kitchen towels).

4. **Health Care**
   - Over-the-counter medications, vitamins, and dietary supplements.

5. **Consumables**
   - Items like batteries and packaging materials.

FMCG products are characterized by their low margin but high volume sales, and they typically have a quick turnover rate in stores.


In [6]:
# Step 5b: Run the agent on a query that REQUIRES real-time web search
#
# This query asks for recent news and specific company examples —
# information the LLM cannot reliably provide from its training data alone.
#
# Expected agent loop:
#   Turn 1: LLM reads the query → decides to call search_web
#   Tool:   search_web runs → returns a list of article titles and URLs
#   Turn 2: LLM reads the search results → composes a grounded final answer
#
# This two-turn loop is handled automatically by Runner.run().

realtime_query = (
    "What are the top 3 most recent AI-driven innovations in FMCG supply chains? "
    "Include company names and examples."
)

result = await Runner.run(research_agent, realtime_query)

print("[Query]:", realtime_query)
print("\n[Agent Response]:")
print(result.final_output)


[Query]: What are the top 3 most recent AI-driven innovations in FMCG supply chains? Include company names and examples.

[Agent Response]:
Here are the top three recent AI-driven innovations in FMCG supply chains, along with relevant company examples:

1. **Generative AI for Demand Forecasting**
   - **Company**: Unilever
   - **Example**: Unilever has implemented generative AI models to improve demand forecasting accuracy across its extensive product range. By analyzing past sales data and incorporating external variables like weather and local events, Unilever aims to reduce stockouts and optimize inventory management.

2. **AI-Driven Logistics Optimization**
   - **Company**: Procter & Gamble (P&G)
   - **Example**: P&G utilizes AI algorithms to enhance logistics and supply chain efficiency. This includes real-time route optimization for delivery trucks, which has significantly reduced transportation costs and improved delivery times. Their system analyzes traffic patterns, deliver

In [7]:
# Step 6: Inspect the full run — see every step the agent took
#
# result.new_items is a list of every item produced during the run.
# Common item types:
#   MessageOutputItem  — a text message produced by the LLM
#   ToolCallItem       — the LLM's decision to call a tool (includes arguments)
#   ToolCallOutputItem — the result returned by the tool function
#
# Comparing the steps for Step 5a vs 5b shows whether search_web was invoked.

print("[Steps taken by the agent]:")
for i, item in enumerate(result.new_items, 1):
    print(f"  Step {i}: {type(item).__name__}")


[Steps taken by the agent]:
  Step 1: ToolCallItem
  Step 2: ToolCallOutputItem
  Step 3: ToolCallItem
  Step 4: ToolCallItem
  Step 5: ToolCallOutputItem
  Step 6: ToolCallOutputItem
  Step 7: ToolCallItem
  Step 8: ToolCallItem
  Step 9: ToolCallOutputItem
  Step 10: ToolCallOutputItem
  Step 11: ToolCallItem
  Step 12: ToolCallItem
  Step 13: ToolCallOutputItem
  Step 14: ToolCallOutputItem
  Step 15: ToolCallItem
  Step 16: ToolCallItem
  Step 17: ToolCallOutputItem
  Step 18: ToolCallOutputItem
  Step 19: ToolCallItem
  Step 20: ToolCallOutputItem
  Step 21: MessageOutputItem


#### Summary

In this demo you built a tool-enabled agent using the **OpenAI Agents SDK**:

- Configured the SDK with a standard **OpenAI** async client
- Defined a `search_web` tool with the `@function_tool` decorator — the docstring is automatically used as the tool description the LLM sees
- Created an `Agent` with a system prompt (`instructions`), a model, and a tool list
- Ran the agent with `Runner.run()` on two queries — one answered from the model's own knowledge, one triggering a live Tavily web search
- Inspected `result.new_items` to see every LLM turn and tool call in the run

The key insight: the agent decides **autonomously** when to call a tool. You define the tools and instructions; the LLM reasons about which tools are needed for each query.